In [1]:
import os
import pandas as pd
import numpy as np
from datasets import load_dataset, concatenate_datasets,Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import (
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)
from sklearn.model_selection import StratifiedKFold
from collections import Counter

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CEFR_LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
label2id = {lvl: i for i,lvl in enumerate(CEFR_LEVELS)}

In [3]:
# Load all English datasets
english_datasets = [
    load_dataset("UniversalCEFR/readme_en")["train"],
    load_dataset("UniversalCEFR/cefr_asag_en")["train"],
    load_dataset("UniversalCEFR/icle500_en")["train"],
    load_dataset("UniversalCEFR/cefr_sp_en")["train"],
    load_dataset("UniversalCEFR/elg_cefr_en")["train"],
    load_dataset("UniversalCEFR/cambridge_exams_en")["train"],
]
english_data = concatenate_datasets(english_datasets)

In [4]:
english_data

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 14663
})

In [5]:
# Filter to keep only valid CEFR levels
filtered_data = english_data.filter(lambda x: x["cefr_level"] in CEFR_LEVELS)

In [6]:
# Remove duplicate texts
df = filtered_data.to_pandas().drop_duplicates(subset="text", keep="first")
filtered_data = Dataset.from_pandas(df)

In [7]:
filtered_data

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text', '__index_level_0__'],
    num_rows: 14210
})

In [6]:
model_name = "EuroBERT/EuroBERT-210m"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
data_collator = DataCollatorWithPadding(tokenizer)

In [7]:
def preprocess(batch):
    toks = tokenizer(batch["text"], truncation=True, max_length=256)
    toks["labels"] = [label2id[l] for l in batch["cefr_level"]]
    return toks

In [8]:
# Tokenize the dataset
tokenized_data = filtered_data.map(preprocess, batched=True, remove_columns=filtered_data.column_names)

# Split English into train/val
n = len(tokenized_data)
train_end = int(0.8 * n)
dev_end = int(0.9 * n)

ds_train = tokenized_data.select(range(0, train_end))
ds_dev   = tokenized_data.select(range(train_end, dev_end))
ds_test  = tokenized_data.select(range(dev_end, n))

Map: 100%|██████████| 14210/14210 [00:01<00:00, 7968.31 examples/s] 


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(CEFR_LEVELS), trust_remote_code=True)

Some weights of EuroBertForSequenceClassification were not initialized from the model checkpoint at EuroBERT/EuroBERT-210m and are newly initialized: ['classifier.bias', 'classifier.weight', 'dense.bias', 'dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, labels=list(range(len(CEFR_LEVELS))), zero_division=0
    )

    metrics = {}
    for i, label in enumerate(CEFR_LEVELS):
        metrics[f"{label}_precision"] = precision[i]
        metrics[f"{label}_recall"] = recall[i]
        metrics[f"{label}_f1"] = f1[i]

    metrics["eval_accuracy"] = accuracy_score(labels, preds)
    metrics["eval_weighted_f1"] = f1_score(labels, preds, average="weighted")
    metrics["eval_weighted_precision"] = precision_score(labels, preds, average="weighted")
    metrics["eval_weighted_recall"] = recall_score(labels, preds, average="weighted")
    return metrics

In [11]:
# Training args
args = TrainingArguments(
    output_dir="./eurobert_cefr_english_only",  
    num_train_epochs=3, 
    per_device_train_batch_size=2,              
    per_device_eval_batch_size=3,                
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_weighted_f1",
    greater_is_better=True,
    seed=42,
    learning_rate=3.6e-5,
    warmup_ratio=0.1,
    gradient_accumulation_steps=16,      
    optim="adamw_torch_fused",                   
    lr_scheduler_type="linear",                  
    adam_beta1=0.9,
    adam_beta2=0.999,
    adam_epsilon=1e-8,
    save_total_limit=1,
)

# Trainer setup
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,  
)

C:\Users\c24082331\AppData\Local\Temp\ipykernel_12772\3133241838.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [12]:
# Train on English-only data
trainer.train()

# Evaluate on Dev set
trainer.evaluate()

# Save model, tokenizer, and trainer state
save_dir = "./eurobert_cefr_english_only/final_model"
trainer.save_model(save_dir)                    
tokenizer.save_pretrained(save_dir)            
trainer.state.save_to_json(os.path.join(save_dir, "trainer_state.json"))  

print(f"Model saved to {save_dir}")

Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1,Weighted Precision,Weighted Recall,A1 Precision,A1 Recall,A1 F1,A2 Precision,A2 Recall,A2 F1,B1 Precision,B1 Recall,B1 F1,B2 Precision,B2 Recall,B2 F1,C1 Precision,C1 Recall,C1 F1,C2 Precision,C2 Recall,C2 F1
1,1.094100,1.572998,0.360310,0.295293,0.362097,0.360310,0.500000,0.100000,0.166667,0.214286,0.285714,0.244898,0.603604,0.638095,0.620370,0.297409,0.810734,0.435178,0.411765,0.126697,0.193772,0.000000,0.000000,0.000000
2,0.766900,1.306221,0.546798,0.519599,0.512217,0.546798,1.000000,0.400000,0.571429,0.400000,0.571429,0.470588,0.581590,0.661905,0.619154,0.416961,0.666667,0.513043,0.663230,0.582202,0.620080,0.000000,0.000000,0.000000
3,0.429000,1.801105,0.520056,0.492332,0.496787,0.520056,1.000000,0.500000,0.666667,0.379310,0.523810,0.440000,0.602510,0.685714,0.641425,0.397781,0.709040,0.509645,0.634429,0.494721,0.555932,0.000000,0.000000,0.000000


c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Model saved to ./eurobert_cefr_english_only/final_model
